# Reproduce the 500-route report

This notebook rebuilds the report tables and charts from the committed CSV outputs. It does not rerun the optimizer.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

current_dir = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (current_dir, *current_dir.parents)
                     if (p / "README.md").is_file() and (p / "notebooks").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from inside the cloned project folder.")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROCESSED_DIR / "github_figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sample_df = pd.read_csv(PROCESSED_DIR / "evaluation_routes_500.csv")
outcomes_df = pd.read_csv(PROCESSED_DIR / "optimization_500_final_outcomes.csv")
paired_df = pd.read_csv(PROCESSED_DIR / "optimization_500_paired_comparison.csv")
published_summary_df = pd.read_csv(PROCESSED_DIR / "optimization_500_final_objective_summary.csv")
published_strata_df = pd.read_csv(PROCESSED_DIR / "optimization_500_strata_solve_rates.csv")

print("Project root:", PROJECT_ROOT)
print("Sample routes:", len(sample_df))
print("Final route-objective records:", len(outcomes_df))
print("Paired routes:", len(paired_df))

In [ ]:
def parse_bool_column(series, column_name, allow_missing=False):
    normalized = series.astype("string").str.strip().str.lower()
    parsed = normalized.map({"true": True, "false": False, "1": True, "0": False})
    unexpected = series.notna() & parsed.isna()
    if unexpected.any():
        raise ValueError(f"Unexpected values in {column_name}: {series[unexpected].unique().tolist()}")
    if allow_missing:
        return parsed.astype("boolean")
    if parsed.isna().any():
        raise ValueError(f"Missing values found in {column_name}")
    return parsed.astype(bool)

sample_df["route_id"] = sample_df["route_id"].astype(str)
outcomes_df["route_id"] = outcomes_df["route_id"].astype(str)
paired_df["route_id"] = paired_df["route_id"].astype(str)
outcomes_df["is_feasible"] = parse_bool_column(outcomes_df["is_feasible"], "is_feasible")
outcomes_df["all_stops_visited_once"] = parse_bool_column(
    outcomes_df["all_stops_visited_once"], "all_stops_visited_once", allow_missing=True
)

feasible_df = outcomes_df.loc[outcomes_df["is_feasible"]].copy()
if feasible_df["all_stops_visited_once"].isna().any():
    raise ValueError("Feasible outcomes have missing all_stops_visited_once values")
if not feasible_df["all_stops_visited_once"].all():
    raise ValueError("At least one feasible outcome did not visit every stop once")
if feasible_df["optimized_window_violations"].fillna(0).ne(0).any():
    raise ValueError("A feasible outcome has time-window violations")

sampled_n = sample_df["route_id"].nunique()
# Calculate elapsed minutes saved and wait directly from historical and optimized values.
feasible_df["elapsed_minutes_saved"] = feasible_df["historical_elapsed_minutes"] - feasible_df["optimized_elapsed_minutes"]
objective_summary = feasible_df.groupby("objective").agg(
    feasible_routes=("route_id", "nunique"),
    average_driving_reduction_percent=("travel_reduction_percent", "mean"),
    median_driving_reduction_percent=("travel_reduction_percent", "median"),
    average_elapsed_minutes_saved=("elapsed_minutes_saved", "mean"),
    median_elapsed_minutes_saved=("elapsed_minutes_saved", "median"),
    routes_faster_than_historical=("elapsed_minutes_saved", lambda s: int((s > 0).sum())),
    average_wait_minutes=("optimized_wait_minutes", "mean"),
    total_window_violations=("optimized_window_violations", "sum"),
    routes_visiting_every_stop_once=("all_stops_visited_once", "sum"),
).reset_index()
objective_summary["attempted_routes"] = sampled_n
objective_summary["solve_rate_percent"] = 100 * objective_summary["feasible_routes"] / sampled_n

pivot = feasible_df.pivot(index="route_id", columns="objective", values=["optimized_elapsed_minutes", "optimized_travel_minutes"])
paired = pivot.dropna(subset=[("optimized_elapsed_minutes", "travel"), ("optimized_elapsed_minutes", "elapsed")]).copy()
paired["elapsed_change_minutes"] = paired[("optimized_elapsed_minutes", "elapsed")] - paired[("optimized_elapsed_minutes", "travel")]
paired["drive_change_minutes"] = paired[("optimized_travel_minutes", "elapsed")] - paired[("optimized_travel_minutes", "travel")]
paired_summary = pd.DataFrame([{
    "routes_feasible_under_both_objectives": len(paired),
    "routes_elapsed_objective_faster": int((paired["elapsed_change_minutes"] < 0).sum()),
    "mean_elapsed_change_minutes": paired["elapsed_change_minutes"].mean(),
    "median_elapsed_change_minutes": paired["elapsed_change_minutes"].median(),
    "mean_drive_change_minutes": paired["drive_change_minutes"].mean(),
    "median_drive_change_minutes": paired["drive_change_minutes"].median(),
}])

route_feasibility = (feasible_df.groupby(["route_id", "objective"])["is_feasible"].any()
                     .unstack(fill_value=False).reindex(columns=["travel", "elapsed"], fill_value=False))
strata_base = sample_df.merge(route_feasibility, on="route_id", how="left").fillna({"travel": False, "elapsed": False})
strata_summary = strata_base.groupby(["size_quartile", "window_density_quartile"]).agg(
    sampled_routes=("route_id", "nunique"), travel_feasible_routes=("travel", "sum"),
    elapsed_feasible_routes=("elapsed", "sum"),
).reset_index()
strata_summary["paired_feasible_routes"] = strata_base.assign(
    paired=strata_base["travel"] & strata_base["elapsed"]
).groupby(["size_quartile", "window_density_quartile"])["paired"].sum().to_numpy()
for obj in ["travel", "elapsed"]:
    strata_summary[f"{obj}_solve_rate_percent"] = 100 * strata_summary[f"{obj}_feasible_routes"] / strata_summary["sampled_routes"]
strata_summary["paired_solve_rate_percent"] = 100 * strata_summary["paired_feasible_routes"] / strata_summary["sampled_routes"]

print("Routes in sample:", sampled_n)
print("Feasible route-objective results:", len(feasible_df))
print("Routes feasible under both objectives:", len(paired))
print("Routes unresolved under both objectives:", sampled_n - len(paired))
print("\nObjective summary:")
display(objective_summary.round(2))
print("\nPaired objective comparison:")
display(paired_summary.round(2))
print("\nSolve rates by size and window-density quartile:")
display(strata_summary.round(2))
print("\nPublished summary files loaded for reference:")
display(published_summary_df.head())
display(published_strata_df.head())

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
summary_plot = objective_summary.set_index("objective").reindex(["travel", "elapsed"])
colors = ["#2F6B9A", "#E07A3F"]
axes[0].bar(summary_plot.index, summary_plot["average_driving_reduction_percent"], color=colors)
axes[0].set_title("Average driving reduction")
axes[0].set_ylabel("Percent")
axes[1].bar(summary_plot.index, summary_plot["average_elapsed_minutes_saved"], color=colors)
axes[1].set_title("Average elapsed time saved")
axes[1].set_ylabel("Minutes")
axes[2].bar(summary_plot.index, summary_plot["solve_rate_percent"], color=colors)
axes[2].set_title("Feasible solve rate")
axes[2].set_ylabel("Percent")
axes[2].set_ylim(0, 100)
fig.suptitle("500-route evaluation: objective comparison", fontsize=14)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "objective_summary.png", dpi=180, bbox_inches="tight")
plt.show()

paired_changes = paired["elapsed_change_minutes"].dropna()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(paired_changes, bins=30, color="#2F6B9A", edgecolor="white")
ax.axvline(0, color="#D04A3A", linestyle="--", linewidth=1.5)
ax.axvline(paired_changes.median(), color="#E07A3F", linewidth=1.5, label=f"Median: {paired_changes.median():.1f} min")
ax.set(title="Elapsed-objective change vs travel objective", xlabel="Change in elapsed time (minutes; negative is faster)", ylabel="Routes")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "paired_elapsed_change.png", dpi=180, bbox_inches="tight")
plt.show()

heat = strata_summary.pivot(index="size_quartile", columns="window_density_quartile", values="paired_solve_rate_percent")
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(heat, cmap="YlGnBu", vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(len(heat.columns)), labels=heat.columns)
ax.set_yticks(range(len(heat.index)), labels=heat.index)
ax.set(xlabel="Window-density quartile", ylabel="Route-size quartile", title="Paired feasible solve rate by route group")
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        value = heat.iloc[i, j]
        if pd.notna(value): ax.text(j, i, f"{value:.0f}%", ha="center", va="center", color="black")
fig.colorbar(im, ax=ax, label="Solve rate (%)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "solve_rate_by_group.png", dpi=180, bbox_inches="tight")
plt.show()
print("Charts saved in:", FIGURES_DIR)